# পাঠ ০৫ - এজেন্টিক RAG


## সেটআপ

এই নোটবুকটি Microsoft Agent Framework ব্যবহার করে Agentic RAG (Retrieval-Augmented Generation) প্যাটার্ন প্রদর্শন করে।

**প্রয়োজনীয়তা:**
- `AZURE_SEARCH_SERVICE_ENDPOINT` — আপনার Azure AI Search সার্ভিস এন্ডপয়েন্ট
- `AZURE_SEARCH_API_KEY` — আপনার Azure AI Search API কী
- পরিবেশ পরিবর্তনশীলের মাধ্যমে কনফিগার করা Azure OpenAI ডিপ্লয়মেন্ট
- Azure CLI প্রমাণীকৃত (`az login`)


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Agentic RAG কী?

প্রচলিত RAG একটি নির্দিষ্ট ধাপ অনুসরণ করে: ডকুমেন্ট পুনরুদ্ধার করা, তারপর একটি প্রতিক্রিয়া তৈরি করা। **Agentic RAG** আরও এগিয়ে যায় এজেন্টকে স্বায়ত্তশাসন দিয়ে **কবে** এবং **কিভাবে** তথ্য পুনরুদ্ধার করতে হবে তা সিদ্ধান্ত নিতে।

Agentic RAG-এর মাধ্যমে এজেন্ট করতে পারে:
- **সিদ্ধান্ত নেওয়া** যে প্রশ্নের উত্তর দেওয়ার আগে পুনরুদ্ধার প্রয়োজন কিনা
- **বেছে নেওয়া** কোন ডেটা সোর্স বা টুল থেকে প্রশ্ন করা হবে
- **মূল্যায়ন করা** পুনরুদ্ধারকৃত ফলাফল এবং যদি প্রথম প্রচেষ্টা পর্যাপ্ত না হয় তবে ফলো-আপ পুনরুদ্ধার করা
- **সংযোগ করা** একাধিক পুনরুদ্ধার ধাপের তথ্য একটি সঙ্গতিপূর্ণ উত্তরে

এইভাবে এজেন্টটি একটি স্থির পুনরুদ্ধার-তারপরে-উৎপাদন ধাপের তুলনায় আরও নমনীয় এবং সঠিক হয়।


## একটি অনুসন্ধান সরঞ্জাম তৈরি করা

Agentic RAG-এ, বাহ্যিক ডেটা উৎসগুলি **সরঞ্জাম** হিসাবে মোড়ানো হয় যা এজেন্ট প্রয়োজনে আহ্বান করতে পারে। এটি এজেন্টকে অনুসন্ধানকে আরও একটি কার্যকলাপ হিসাবে বিবেচনা করতে দেয়, একটি বাধ্যতামূলক ধাপ না হয়ে।

নিচে আমরা একটি ভ্রমণ জ্ঞানের ভিত্তি সংজ্ঞায়িত করি এবং এটি একটি সরঞ্জাম হিসাবে প্রকাশ করি যা এজেন্ট গন্তব্য তথ্য খুঁজতে কল করতে পারে।


In [ ]:
TRAVEL_KNOWLEDGE_BASE = {
    "Barcelona": "Barcelona is Spain's cosmopolitan capital of Catalonia. Best visited Mar-May or Sep-Nov. Known for Gaudí architecture, La Rambla, beaches. Average daily cost: $150-200.",
    "Tokyo": "Tokyo is Japan's capital, mixing ultramodern with traditional. Best visited Mar-Apr (cherry blossoms) or Oct-Nov. Known for Shibuya, temples, sushi. Average daily cost: $200-250.",
    "Paris": "Paris is France's capital and a global center for art, fashion, and culture. Best visited Apr-Jun or Sep-Oct. Known for Eiffel Tower, Louvre, cuisine. Average daily cost: $180-250.",
    "Cape Town": "Cape Town sits on South Africa's southwest tip. Best visited Nov-Mar. Known for Table Mountain, wine regions, wildlife. Average daily cost: $100-150.",
}


@tool(approval_mode="never_require")
def search_travel_knowledge(
    query: Annotated[str, "The search query about a travel destination"]
) -> str:
    """Search the travel knowledge base for destination information."""
    results = []
    for destination, info in TRAVEL_KNOWLEDGE_BASE.items():
        if query.lower() in destination.lower() or any(
            word in info.lower() for word in query.lower().split()
        ):
            results.append(f"**{destination}**: {info}")
    return (
        "\n\n".join(results)
        if results
        else "No matching destinations found in the knowledge base."
    )

## RAG এজেন্ট তৈরি করা

এখন আমরা একটি এজেন্ট তৈরি করছি যা নির্দেশিত যে **সর্বদা উত্তর দেওয়ার আগে তথ্য আহরণ করবে**। এজেন্টটি তার উত্তরগুলিকে নিজের প্রশিক্ষণ ডাটার উপর নির্ভর না করে জ্ঞানের ভিত্তিতে স্থাপন করার জন্য `search_travel_knowledge` টুলটি ব্যবহার করে।


In [ ]:
agent = await provider.create_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGAgent",
    instructions="""You are a knowledgeable travel advisor. Before answering questions about destinations:
1. ALWAYS search the travel knowledge base first
2. Base your answers on retrieved information
3. If information is not in the knowledge base, say so clearly
4. Provide specific details like costs, best seasons, and highlights.""",
)

response = await agent.run(
    "I'm interested in visiting somewhere with great architecture. What destinations would you recommend?",
    )
print(response)

## পুনরাবৃত্তিমূলক অনুসন্ধান — মেকার-চেকার প্যাটার্ন

Agentic RAG এর একটি প্রধান সুবিধা হল **পুনরাবৃত্তিমূলক অনুসন্ধান**। এজেন্ট তার প্রাথমিক অনুসন্ধানগুলিকে যাচাই, পরিমার্জন বা সম্প্রসারণ করার জন্য একাধিক রাউন্ড অনুসন্ধান করতে পারে — যা "মেকার-চেকার" কর্মপ্রবাহের মতো:

1. **মেকার ধাপ**: এজেন্ট প্রাথমিক তথ্য উদ্ধার করে এবং একটি উত্তর খসড়া তৈরি করে।
2. **চেকার ধাপ**: এজেন্ট তথ্য যাচাই বা ঘাটতি পূরণের জন্য অতিরিক্ত অনুসন্ধান করে।

নিচে, এজেন্টকে একটি প্রশ্ন জানানো হয়েছে যা একাধিক গন্তব্য তুলনা করতে হবে, যা এটিকে কয়েকবার অনুসন্ধান করতে প্ররোচিত করে।


In [ ]:
checker_agent = await provider.create_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGCheckerAgent",
    instructions="""You are a meticulous travel advisor who double-checks recommendations.
When answering travel questions:
1. Search for relevant destinations first
2. For each destination found, search again with the destination name to get full details
3. Compare the options using verified information
4. Present a final recommendation with specific costs, best travel times, and highlights
5. If any detail seems incomplete, search once more to confirm before responding.""",
)

response = await checker_agent.run(
    "I have a $175/day budget and want to travel in April. Which destinations fit my budget and timing?",
    )
print(response)

## সারাংশ

এই পাঠে আপনি শিখেছেন কিভাবে Microsoft Agent Framework ব্যবহার করে একটি **Agentic RAG** সিস্টেম তৈরি করা যায়:

- **Agentic RAG** এজেন্টদের স্বতন্ত্রভাবে সিদ্ধান্ত নেওয়ার ক্ষমতা দেয় কখন তথ্য আহরণ করতে হবে, ফলে আহরণ স্থির নয়, গতিশীল হয়।
- **টুল হিসেবে ডেটা সোর্স**: বাহ্যিক জ্ঞানভাণ্ডার (যেমন Azure AI Search) টুল হিসেবে মোড়ানো হয় যা এজেন্ট আহ্বান করতে পারে।
- **পুনরাবৃত্তিমূলক আহরণ**: মেকার-চেকার প্যাটার্ন এজেন্টকে একাধিক আহরণ পর্যায় সম্পাদন করার সুযোগ দেয় — অনুসন্ধান, যাচাই, এবং পরিমার্জন করে চূড়ান্ত উত্তর প্রদান করার আগে।

প্রোডাকশনে, আপনি ইন-মেমরি `TRAVEL_KNOWLEDGE_BASE` এর পরিবর্তে একটি বাস্তব Azure AI Search ইনডেক্স ব্যবহার করবেন বৃহৎ পরিসরের ভ্রমণ নথি আহরণের জন্য।


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**বাতিল ঘোষণা**:  
এই দলিলটি AI অনুবাদ সেবা [Co-op Translator](https://github.com/Azure/co-op-translator) ব্যবহার করে অনূদিত হয়েছে। আমরা যথাসাধ্য সঠিকতার চেষ্টা করি, তবে স্বয়ংক্রিয় অনুবাদে ভুল বা অমিল থাকার সম্ভাবনা রয়েছে তা দয়া করে মনে রাখুন। মূল ভাষার দস্তাবেজটিকেই কর্তৃত্বপূর্ণ উৎস হিসেবে বিবেচনা করা উচিত। গুরুত্বপূর্ণ তথ্যের ক্ষেত্রে পেশাদার মানব অনুবাদের পরামর্শ দেওয়া হয়। এই অনুবাদের ব্যবহারে কোনো ভুল বোঝাবুঝি বা ভুল ব্যাখ্যার জন্য আমরা দায়ী নই।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
